In [1]:
!pip install entsoe-py requests pandas
!pip install matplotlib seaborn scikit-learn
import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient

import os
from dotenv import load_dotenv

import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import pandas as pd
# import requests

# # Hämta t.ex. ett helt år (2024) från Nord Pool-spegeln
# # Format: https://www.elprisetjustnu.se/api/v1/prices/{år}/{månad}-{dag}_SE3.json
# dates = pd.date_range("2024-01-01", "2024-01-05")  # Ändra spann efter behov
# all_prices = []

# for date in dates:
#     url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.strftime('%Y/%m-%d')}_SE3.json"
#     r = requests.get(url)
#     if r.status_code == 200:
#         all_prices.extend(r.json())

# df_prices = pd.DataFrame(all_prices)
# print(df_prices)
# df_prices = df_prices[["time_start", "EUR_per_kWh"]].rename(
#     columns={"time_start": "timestamp", "EUR_per_kWh": "spot_price_eur_kwh"}
# )
# df_prices["spot_price_eur_mwh"] = df_prices["spot_price_eur_kwh"] * 1000
# df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"])

# #print(df_prices.head())



In [ ]:
# ==========================================
# DEL 1: HÄMTA ELPRISER FRÅN ENTSO-E
# ==========================================
import os
import time
import pandas as pd
from dotenv import load_dotenv
from entsoe import EntsoePandasClient

# 1. Konfiguration
load_dotenv(override=True)
API_KEY = os.getenv("ENTSOE_API_KEY")
if not API_KEY:
    raise ValueError("Hittade ingen ENTSOE_API_KEY i din .env-fil!")

# Tvätta nyckeln ren från dolda mellanslag och eventuella citattecken
API_KEY = API_KEY.strip().strip('"').strip("'")

ZONES = {"SE1": "SE_1", "SE2": "SE_2", "SE3": "SE_3", "SE4": "SE_4"}

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"
CACHE_DIR = "cache_entsoe"

os.makedirs(CACHE_DIR, exist_ok=True)
client = EntsoePandasClient(api_key=API_KEY)

# 2. Hämta zonvis i månadsvisa block (1MS) för att undvika 504 och 599 timeout
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="1MS", tz="UTC")
zone_price_dfs = []

for zone_name, zone_code in ZONES.items():
    print(f"\nBehandlar elpriser för {zone_name}...")
    zone_blocks = []

    for i in range(len(date_range) - 1):
        start_ts = date_range[i]
        end_ts = date_range[i + 1]

        # Månadsvisa filnamn
        cache_file = os.path.join(
            CACHE_DIR,
            f"price_{zone_name}_{start_ts.strftime('%Y%m')}.csv",
        )

        # Läs från lokal cache om den redan finns
        if os.path.exists(cache_file):
            df_cached = pd.read_csv(cache_file)
            df_cached["timestamp"] = pd.to_datetime(df_cached["timestamp"])
            zone_blocks.append(df_cached)
            print(f"  [{zone_name}] Cache hittad: {start_ts.strftime('%Y-%m')}")
            continue

        # Hämta från API med upp till 3 försök vid timeout
        success = False
        for attempt in range(1, 4):
            try:
                print(
                    f"  [{zone_name}] Hämtar {start_ts.strftime('%Y-%m')} (försök {attempt}/3)..."
                )
                series = client.query_day_ahead_prices(
                    zone_code, start=start_ts, end=end_ts
                )
                df_block = series.reset_index()
                df_block.columns = ["timestamp", f"price_{zone_name.lower()}_eur_mwh"]
                df_block["timestamp"] = pd.to_datetime(
                    df_block["timestamp"]
                ).dt.tz_convert("UTC")

                df_block.to_csv(cache_file, index=False)
                zone_blocks.append(df_block)
                print(f"     -> Sparad ({len(df_block)} rader)")
                success = True
                break
            except Exception as e:
                print(f"     Varning: {e}")
                time.sleep(3 * attempt)
            time.sleep(0.5)

        if not success:
            print(f"  Kunde inte hämta {zone_name} för {start_ts.strftime('%Y-%m')}")

    if zone_blocks:
        df_zone = pd.concat(zone_blocks, ignore_index=True).drop_duplicates(
            subset=["timestamp"]
        )
        zone_price_dfs.append(df_zone)

# 3. Slå ihop alla zonpriser och spara
if zone_price_dfs:
    df_all_prices = zone_price_dfs[0]
    for df_p in zone_price_dfs[1:]:
        df_all_prices = pd.merge(df_all_prices, df_p, on="timestamp", how="outer")

    df_all_prices = df_all_prices.sort_values(by="timestamp").reset_index(drop=True)
    df_all_prices.to_csv("dataset/prices_all_zones.csv", index=False)

    print("\n--- DEL 1 KLAR ---")
    print(f"Sparade {len(df_all_prices)} rader till 'prices_all_zones.csv'")
    display(df_all_prices.head())
else:
    print("\nIngen prisdata kunde hämtas eller läsas in.")


Behandlar elpriser för SE1...
  [SE1] Cache hittad: 2025-01
  [SE1] Cache hittad: 2025-02
  [SE1] Cache hittad: 2025-03
  [SE1] Cache hittad: 2025-04
  [SE1] Cache hittad: 2025-05
  [SE1] Cache hittad: 2025-06
  [SE1] Cache hittad: 2025-07
  [SE1] Cache hittad: 2025-08
  [SE1] Cache hittad: 2025-09
  [SE1] Cache hittad: 2025-10
  [SE1] Cache hittad: 2025-11
  [SE1] Cache hittad: 2025-12

Behandlar elpriser för SE2...
  [SE2] Cache hittad: 2025-01
  [SE2] Cache hittad: 2025-02
  [SE2] Cache hittad: 2025-03
  [SE2] Cache hittad: 2025-04
  [SE2] Hämtar 2025-05 (försök 1/3)...
     -> Sparad (745 rader)
  [SE2] Hämtar 2025-06 (försök 1/3)...
     -> Sparad (721 rader)
  [SE2] Hämtar 2025-07 (försök 1/3)...
     -> Sparad (745 rader)
  [SE2] Hämtar 2025-08 (försök 1/3)...
     -> Sparad (745 rader)
  [SE2] Hämtar 2025-09 (försök 1/3)...
     -> Sparad (727 rader)
  [SE2] Hämtar 2025-10 (försök 1/3)...
     -> Sparad (2977 rader)
  [SE2] Hämtar 2025-11 (försök 1/3)...
     -> Sparad (2881 r

,timestamp,price_se1_eur_mwh,price_se2_eur_mwh,price_se3_eur_mwh,price_se4_eur_mwh
0,2025-01-01 00:00:00+00:00,3.97,3.69,2.47,2.04
1,2025-01-01 01:00:00+00:00,3.82,3.55,2.44,2.04
2,2025-01-01 02:00:00+00:00,3.49,3.19,1.98,1.54
3,2025-01-01 03:00:00+00:00,3.46,3.15,1.79,1.18
4,2025-01-01 04:00:00+00:00,3.89,3.68,2.71,2.38


In [ ]:
import requests
import pandas as pd

ZONE_COORDS = {
    "SE1": {"lat": 65.5848, "lon": 22.1567, "city": "Luleå"},
    "SE2": {"lat": 62.3908, "lon": 17.3069, "city": "Sundsvall"},
    "SE3": {"lat": 59.3293, "lon": 18.0686, "city": "Stockholm"},
    "SE4": {"lat": 55.6050, "lon": 13.0038, "city": "Malmö"},
}

weather_frames = []
weather_url = "https://archive-api.open-meteo.com/v1/archive"

for zone_name, conf in ZONE_COORDS.items():
    print(f"Hämtar väder för {zone_name} ({conf['city']})...")
    params = {
        "latitude": conf["lat"],
        "longitude": conf["lon"],
        "start_date": "2025-01-01",
        "end_date": "2025-12-31",
        "hourly": "temperature_2m,wind_speed_10m,rain",
        "timezone": "UTC",
    }
    res = requests.get(weather_url, params=params).json()
    hourly = res["hourly"]

    df_w = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(hourly["time"]).tz_localize("UTC"),
            f"temp_{zone_name.lower()}_c": hourly["temperature_2m"],
            f"wind_{zone_name.lower()}_kmh": hourly["wind_speed_10m"],
            f"rain_{zone_name.lower()}_mm": hourly["rain"],
        }
    )
    weather_frames.append(df_w)

df_all_weather = weather_frames[0]
for df_w in weather_frames[1:]:
    df_all_weather = pd.merge(df_all_weather, df_w, on="timestamp", how="inner")

df_all_weather.to_csv("dataset/weather_all_zones.csv", index=False)
print(f"\nKlart! Sparade {len(df_all_weather)} rader till 'weather_all_zones.csv'.")

Hämtar väder för SE1 (Luleå)...
Hämtar väder för SE2 (Sundsvall)...
Hämtar väder för SE3 (Stockholm)...
Hämtar väder för SE4 (Malmö)...

Klart! Sparade 8760 rader till 'weather_all_zones.csv'.


In [ ]:
import pandas as pd

# 1. Läs in de två del-dataseten
print("Läser in del-dataset...")
df_prices = pd.read_csv("prices_all_zones.csv")
df_weather = pd.read_csv("weather_all_zones.csv")

# 2. Säkerställ att båda tidsstämplarna tolkas som UTC
df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"], utc=True)
df_weather["timestamp"] = pd.to_datetime(df_weather["timestamp"], utc=True)

# 3. Slå ihop på timestamp (inner join behåller timmar där båda finns)
print("Slår ihop väder och elpriser per timme...")
df_master = pd.merge(df_weather, df_prices, on="timestamp", how="inner")

# Sortera kronologiskt
df_master = df_master.sort_values(by="timestamp").reset_index(drop=True)

# Lägg till lokal svensk tid som referens (utmärkt för dygns- och timanalyser)
df_master["timestamp_local"] = df_master["timestamp"].dt.tz_convert("Europe/Stockholm")

# 4. Spara till EN gemensam CSV-fil
output_file = "dataset/all_zones_complete_2025.csv"
df_master.to_csv(output_file, index=False)

print(f"\nKlart! All data samlad i '{output_file}'.")
print(f"Dimensioner: {df_master.shape[0]} rader, {df_master.shape[1]} kolumner.")
display(df_master.head())

Läser in del-dataset...
Slår ihop väder och elpriser per timme...

Klart! All data samlad i 'all_zones_complete_2025.csv'.
Dimensioner: 8760 rader, 18 kolumner.


,timestamp,temp_se1_c,wind_se1_kmh,rain_se1_mm,temp_se2_c,wind_se2_kmh,rain_se2_mm,temp_se3_c,wind_se3_kmh,rain_se3_mm,temp_se4_c,wind_se4_kmh,rain_se4_mm,price_se1_eur_mwh,price_se2_eur_mwh,price_se3_eur_mwh,price_se4_eur_mwh,timestamp_local
0,2025-01-01 00:00:00+00:00,-9.1,8.6,0.0,-13.1,3.2,0.0,-0.6,18.4,0.0,5.5,43.1,0.9,3.97,3.69,2.47,2.04,2025-01-01 01:00:00+01:00
1,2025-01-01 01:00:00+00:00,-11.6,9.2,0.0,-13.4,5.5,0.0,0.2,12.7,0.0,5.9,43.7,0.3,3.82,3.55,2.44,2.04,2025-01-01 02:00:00+01:00
2,2025-01-01 02:00:00+00:00,-12.2,8.3,0.0,-14.6,6.6,0.0,0.6,5.2,0.1,6.0,42.6,0.7,3.49,3.19,1.98,1.54,2025-01-01 03:00:00+01:00
3,2025-01-01 03:00:00+00:00,-12.5,7.3,0.0,-14.3,7.5,0.0,1.2,6.5,0.1,6.3,44.8,0.9,3.46,3.15,1.79,1.18,2025-01-01 04:00:00+01:00
4,2025-01-01 04:00:00+00:00,-12.6,5.7,0.0,-14.5,8.8,0.0,1.5,7.6,0.1,6.7,48.2,0.7,3.89,3.68,2.71,2.38,2025-01-01 05:00:00+01:00


Allt under som hanteras med EDA behöver ändras om

In [ ]:
import pandas as pd

# Läs in den sparade datan (tar bråkdelen av en sekund)
df = pd.read_csv("all_zones_complete_2025.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 1. Kolla saknade värden
print("Saknade värden per kolumn:")
print(df.isna().sum())

# 2. Snabb överblick av min/max och medel
print("\nStatistik:")
print(
    df[["temp_se1_c", "wind_se1_kmh", "rain_mm", "spot_price_eur_mwh"]]
    .describe()
    .round(2)
)

# 3. Snabb korrelationsmatris (se hur mycket vinden pressar priset!)
print("\nKorrelation mot elpris:")
print(df[["temperature_c", "wind_speed_kmh", "rain_mm", "spot_price_eur_mwh"]].corr()["spot_price_eur_mwh"].round(3))

Saknade värden per kolumn:
timestamp            0
temp_se1_c           0
wind_se1_kmh         0
rain_se1_mm          0
temp_se2_c           0
wind_se2_kmh         0
rain_se2_mm          0
temp_se3_c           0
wind_se3_kmh         0
rain_se3_mm          0
temp_se4_c           0
wind_se4_kmh         0
rain_se4_mm          0
price_se1_eur_mwh    0
price_se2_eur_mwh    0
price_se3_eur_mwh    0
price_se4_eur_mwh    0
timestamp_local      0
dtype: int64

Statistik:


KeyError: "['wind_speed_kmh', 'rain_mm', 'spot_price_eur_mwh'] not in index"

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Läs in datan
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 2. Rensa eventuella extrema spikar (t.ex. energikris-toppar > 500 EUR) 
# så att regressionslinjen inte förvrängs helt av enstaka extremvärden
clean_df = df[(df["spot_price_eur_mwh"] >= 0) & (df["spot_price_eur_mwh"] <= 300)].dropna(
    subset=["wind_speed_kmh", "spot_price_eur_mwh"]
)

X = clean_df[["wind_speed_kmh"]]
y = clean_df["spot_price_eur_mwh"]

# 3. Träna linjär regression
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

slope = model.coef_[0]
intercept = model.intercept_
r2 = r2_score(y, y_pred)

print(f"Lutning (koefficient): {slope:.3f} EUR/MWh per km/h vind")
print(f"Intercept: {intercept:.2f} EUR/MWh")
print(f"Förklaringsgrad (R²): {r2:.4f}")

# 4. Skapa visualisering
plt.figure(figsize=(10, 6), dpi=100)

# Eftersom 35 000 punkter blir en gröt kör vi låg alpha (transparens)
sns.regplot(
    data=clean_df.sample(min(5000, len(clean_df))), # Sampla t.ex. 5000 punkter för snabbare och snyggare rendering
    x="wind_speed_kmh",
    y="spot_price_eur_mwh",
    scatter_kws={"alpha": 0.15, "color": "#1f77b4", "s": 15},
    line_kws={"color": "red", "linewidth": 2, "label": f"Trend: y = {slope:.2f}x + {intercept:.1f} (R² = {r2:.3f})"}
)

plt.title("SE3 Elpris vs. Vindhastighet (Linjär Regression)", fontsize=14, fontweight="bold")
plt.xlabel("Vindhastighet vid 10m (km/h)", fontsize=12)
plt.ylabel("Spotpris (EUR/MWh)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Läs in data och extrahera tidsfunktioner
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Säkerställ svensk lokal tid så att morgontoppen hamnar runt kl 07–09 och inte förskjuts av UTC
if df["timestamp"].dt.tz is None:
    df["timestamp"] = df["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Europe/Stockholm")
else:
    df["timestamp"] = df["timestamp"].dt.tz_convert("Europe/Stockholm")

df["hour"] = df["timestamp"].dt.hour
df["day_name"] = df["timestamp"].dt.day_name()
df["day_of_week"] = df["timestamp"].dt.dayofweek  # 0 = Måndag, 6 = Söndag
df["is_weekend"] = df["day_of_week"].isin([5, 6]).map({True: "Helg", False: "Vardag"})

# Sortera dagarna i rätt ordning
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_labels_sv = ["Mån", "Tis", "Ons", "Tor", "Fre", "Lör", "Sön"]

# 2. Skapa figuren med två delgrafer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1]})

# --- GRAF 1: Heatmap (Veckodag vs Timme) ---
pivot_table = df.pivot_table(
    index="day_name", 
    columns="hour", 
    values="spot_price_eur_mwh", 
    aggfunc="mean"
).reindex(day_order)

sns.heatmap(
    pivot_table, 
    cmap="YlOrRd", 
    cbar_kws={"label": "Medelpris (EUR/MWh)"}, 
    ax=ax1,
    annot=False
)
ax1.set_title("Snittpris: Veckodag vs Timme (SE3)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax1.set_ylabel("", fontsize=11)
ax1.set_yticklabels(day_labels_sv, rotation=0)

# --- GRAF 2: Dygnsprofil (Vardag vs Helg) ---
sns.lineplot(
    data=df, 
    x="hour", 
    y="spot_price_eur_mwh", 
    hue="is_weekend", 
    palette={"Vardag": "#d62728", "Helg": "#1f77b4"},
    linewidth=2.5,
    ax=ax2
)
ax2.set_title("Dygnsrytm: Vardag vs Helg", fontsize=13, fontweight="bold")
ax2.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax2.set_ylabel("Spotpris (EUR/MWh)", fontsize=11)
ax2.set_xticks(range(0, 24, 2))
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(title="", frameon=True)

plt.tight_layout()
plt.show()